# pgvector Demo

Use PostgreSQL with pgvector for vector search - ideal when you already have Postgres.

**Prerequisites:**
```bash
# Start Postgres with pgvector
docker-compose -f docker-compose-pgvector.yml up -d

pip install psycopg2-binary sentence-transformers
```

**When to use pgvector:**
- Already have PostgreSQL in your stack
- <100M vectors
- Want single database for relational + vector data

In [1]:
import psycopg2
from sentence_transformers import SentenceTransformer

# Connection settings (match docker-compose.yml)
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "ragdb",
    "user": "raguser",
    "password": "ragpass"
}

# Load embedding model
encoder = SentenceTransformer("all-MiniLM-L6-v2")
VECTOR_DIM = 384

try:
    conn = psycopg2.connect(**DB_CONFIG)
    conn.autocommit = True
    cursor = conn.cursor()
    print("✓ Connected to PostgreSQL")
except Exception as e:
    print(f"✗ Connection failed: {e}")
    print("Run: docker-compose -f docker-compose-pgvector.yml up -d")

✓ Connected to PostgreSQL


---

## 1. Setup Tables

In [2]:
try:
    # Enable pgvector extension
    cursor.execute("CREATE EXTENSION IF NOT EXISTS vector;")
    
    # Create documents table
    cursor.execute(f"""
        CREATE TABLE IF NOT EXISTS documents (
            id SERIAL PRIMARY KEY,
            content TEXT NOT NULL,
            tenant_id VARCHAR(100) NOT NULL,
            embedding vector({VECTOR_DIM})
        );
    """)
    
    # Create HNSW index for fast similarity search
    cursor.execute(f"""
        CREATE INDEX IF NOT EXISTS documents_embedding_idx 
        ON documents USING hnsw (embedding vector_cosine_ops);
    """)
    
    # Create index on tenant_id for filtered queries
    cursor.execute("""
        CREATE INDEX IF NOT EXISTS documents_tenant_idx ON documents(tenant_id);
    """)
    
    print("✓ Tables and indexes created")
except Exception as e:
    print(f"Setup error: {e}")

✓ Tables and indexes created


---

## 2. Insert Documents

In [3]:
documents = [
    ("Remote work is allowed 3 days per week", "acme"),
    ("Vacation policy: 25 days annual leave", "acme"),
    ("Work from home requires manager approval", "globex"),
    ("All employees get 20 days paid leave", "globex"),
]

try:
    # Clear existing data
    cursor.execute("DELETE FROM documents")
    
    for content, tenant_id in documents:
        embedding = encoder.encode(content).tolist()
        cursor.execute(
            "INSERT INTO documents (content, tenant_id, embedding) VALUES (%s, %s, %s)",
            (content, tenant_id, embedding)
        )
    
    print(f"✓ Inserted {len(documents)} documents")
except Exception as e:
    print(f"Insert error: {e}")

✓ Inserted 4 documents


---

## 3. Vector Search

In [4]:
def search(query: str, tenant_id: str = None, limit: int = 3):
    """Search documents with optional tenant filter."""
    query_embedding = encoder.encode(query).tolist()
    
    if tenant_id:
        # Filtered search (multi-tenant)
        cursor.execute("""
            SELECT content, tenant_id, 1 - (embedding <=> %s::vector) as similarity
            FROM documents
            WHERE tenant_id = %s
            ORDER BY embedding <=> %s::vector
            LIMIT %s
        """, (query_embedding, tenant_id, query_embedding, limit))
    else:
        # Unfiltered search
        cursor.execute("""
            SELECT content, tenant_id, 1 - (embedding <=> %s::vector) as similarity
            FROM documents
            ORDER BY embedding <=> %s::vector
            LIMIT %s
        """, (query_embedding, query_embedding, limit))
    
    return cursor.fetchall()

try:
    # Unfiltered search
    query = "How many vacation days?"
    results = search(query)
    print(f"Search: \"{query}\"")
    print("=" * 60)
    for content, tenant, score in results:
        print(f"  [{score:.3f}] {content} (tenant: {tenant})")
    
    # Filtered search
    print(f"\nFiltered (tenant=acme):")
    results = search(query, tenant_id="acme")
    for content, tenant, score in results:
        print(f"  [{score:.3f}] {content}")
except Exception as e:
    print(f"Search error: {e}")

Search: "How many vacation days?"
  [0.680] Vacation policy: 25 days annual leave (tenant: acme)
  [0.451] All employees get 20 days paid leave (tenant: globex)
  [0.367] Remote work is allowed 3 days per week (tenant: acme)

Filtered (tenant=acme):
  [0.680] Vacation policy: 25 days annual leave
  [0.367] Remote work is allowed 3 days per week


---

## Summary

**pgvector Operators:**
- `<=>` : Cosine distance (use `1 - distance` for similarity)
- `<->` : L2 (Euclidean) distance
- `<#>` : Inner product (negative, for max similarity use `-distance`)

**Index Types:**
- `ivfflat`: Faster build, good for <1M vectors
- `hnsw`: Faster search, better for >1M vectors (recommended)

**Use Qdrant when:** You need advanced filtering, high QPS, or >100M vectors